# 교시별 실습 Notebook

d2-p07-08

수업 페이지의 순서대로 실행합니다. 정답 블록과 선택 웹 자동화는 설명을 읽고 별도로 실행하세요.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), *Path.cwd().parents]
course_root = next((p for p in candidates if (p / "raw/기존강사 수업자료").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("수업 자료를 풀고 raw 폴더가 있는 위치에서 Notebook을 여세요.")
os.chdir(course_root)
print("수업 자료 폴더를 확인했습니다.")

# 요약

신용카드 고객 30,000명의 다음 달 부도를 분석하고, 연체 이력과 네 가지 분류 평가 지표를 실습합니다.

## 2일차 7~8교시 — 신용카드 부도 분석과 평가

2026년 10월 7일(수) · 2일차 7~8교시 · 1차 초안. 공식 안내문은 일차별 단원과 총시간을 정하며, 이 페이지의 교시별 묶음은 수업 진행을 위한 제안입니다. 같은 실습을 이어가는 교시는 한 페이지로 묶었습니다.

## 시작 전에

새 Notebook `day2_credit_default`를 만듭니다. Titanic Notebook의 X·y와 혼동하지 않도록 별도 파일을 씁니다. pandas, openpyxl, scikit-learn과 제공 Excel 파일이 필요합니다. 파일 첫 행은 설명이므로 실제 헤더를 두 번째 행에서 읽습니다.

코드 블록 하나를 Notebook의 코드 셀 하나에 붙여 넣고 위에서 아래로 Shift+Enter로 실행합니다. 앞 셀을 실행하지 않고 뒤 셀부터 실행하면 변수를 찾지 못할 수 있습니다. 파일 경로는 수업 폴더 `kpc-lec`를 기준으로 합니다. 현재 위치가 다르면 파일 경로를 실제 파일 위치로 바꿉니다.

## 교시별 진행 제안

- 7교시: 자료 읽기·Target 확인 → 코드형 속성과 금액 변수 구분 → 연체 이력별 부도율·인원 비교.
- 8교시: 기준 모델과 두 학습 모델 비교 → Accuracy·Precision·Recall·F1 → 혼동행렬 → 확률 기준 변화 실습.

## 1. credit card default 데이터 이해 및 target 정의

In [ ]:
from pathlib import Path
import pandas as pd

credit = pd.read_excel(Path("raw/기존강사 수업자료/default of credit card clients.xlsx"), header=1)
target = "default payment next month"
print("행·열:", credit.shape)
print("열:", credit.columns.tolist())
print(credit[target].value_counts().sort_index())
assert credit[target].notna().all()
assert set(credit[target].unique()) == {0, 1}
print(f"부도 비율: {credit[target].mean():.2%}")

Target 1은 다음 달 부도, 0은 부도가 아닙니다. 부도 6,636명, 비부도 23,364명으로 부도 비율은 22.12%입니다. `ID`는 고객 식별 번호이므로 모델 입력에서 제외합니다.

| 열 | 의미 | 처리할 때 확인할 점 |
| --- | --- | --- |
| LIMIT_BAL | 신용 한도 | 원 데이터의 통화 단위는 대만 달러 |
| SEX, EDUCATION, MARRIAGE | 성별·교육·혼인 코드 | 숫자지만 범주 코드 |
| AGE | 나이 | 숫자 변수 |
| PAY_0, PAY_2~PAY_6 | 최근 6개월 상환 상태 | PAY_0가 가장 최근이며 PAY_1 열은 없음 |
| BILL_AMT1~6 | 청구 금액 | 각 월별 숫자 변수 |
| PAY_AMT1~6 | 상환 금액 | 각 월별 숫자 변수 |

상환 상태의 양수 코드는 연체 개월 수를 나타냅니다. 0·음수 코드도 있으므로 양수와 함께 “모두 같은 정상 상태”로 뭉뚱그리지 않습니다. 이 실습의 `has_delay`는 **한 번이라도 1 이상이었는가**만 정의합니다. 변수 정의와 원 연구는 [UCI 원 데이터 설명](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)을 참고합니다.

**직접 실습:** `PAY_0.value_counts().sort_index()`로 코드의 실제 종류를 확인합니다. 부도 여부와 과거 연체 상태가 같은 정답인지 설명합니다.

## 2. 연체 이력 및 고객 속성별 부도율 분석

In [ ]:
pay_columns = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
credit_eda = credit.copy()
credit_eda["has_delay"] = (credit_eda[pay_columns] >= 1).any(axis=1)
print("연체 이력별\n", credit_eda.groupby("has_delay")[target].agg(["count", "sum", "mean"]).round(4))
print("성별 코드별\n", credit_eda.groupby("SEX")[target].agg(["count", "mean"]).round(4))
credit_eda["age_group"] = pd.cut(credit_eda["AGE"], bins=[20, 30, 40, 50, float("inf")],
                               labels=["20s", "30s", "40s", "50+"], right=False)
print("나이대별\n", credit_eda.groupby("age_group", observed=True)[target].agg(["count", "mean"]).round(4))
print("나이 구간 미배정:", credit_eda["age_group"].isna().sum())

각 그룹의 부도율은 Target의 평균입니다. 그룹별 인원도 함께 확인합니다. 특정 속성 그룹의 부도율 차이는 이 자료의 관찰 결과이며 그 속성 자체의 원인 효과나 개별 고객의 정답으로 해석하지 않습니다.

**직접 실습:** 최근 `PAY_0`만 1 이상인 그룹과 최근 6개월 중 한 번이라도 연체한 그룹의 인원·부도율을 비교합니다. 두 조건의 대상 인원이 같아야 하는지 설명합니다.

## 3. 분류 모델 학습 및 평가 지표 이해 (accuracy, precision, recall, F1-score)

0을 항상 예측하면 전체 정확도는 높아도 부도 고객을 찾지 못합니다. Target 1을 양성으로 정의하고 다음 네 지표를 함께 계산합니다.

| 지표 | 질문 |
| --- | --- |
| accuracy | 전체 고객 중 맞힌 비율은? |
| precision | 부도로 예측한 고객 중 실제 부도 비율은? |
| recall | 실제 부도 고객 중 찾아낸 비율은? |
| F1-score | precision과 recall을 함께 요약하면? |

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X = credit.drop(columns=["ID", target])
y = credit[target].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
category_columns = ["SEX", "EDUCATION", "MARRIAGE"]
number_columns = [c for c in X.columns if c not in category_columns]

def credit_preprocessor():
    return ColumnTransformer([
        ("number", Pipeline([("fill", SimpleImputer(strategy="median")),
                             ("scale", StandardScaler())]), number_columns),
        ("category", Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                               ("encode", OneHotEncoder(handle_unknown="ignore"))]), category_columns),
    ])

credit_estimators = {
    "Always non-default": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "RandomForest balanced": RandomForestClassifier(n_estimators=100, max_depth=8,
                              class_weight="balanced", random_state=42, n_jobs=1),
}
credit_models = {}
metric_rows = []
for name, estimator in credit_estimators.items():
    model = Pipeline([("prepare", credit_preprocessor()), ("model", estimator)])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metric_rows.append({"model": name, "accuracy": accuracy_score(y_test, pred),
                        "precision": precision_score(y_test, pred, zero_division=0),
                        "recall": recall_score(y_test, pred, zero_division=0),
                        "F1": f1_score(y_test, pred, zero_division=0)})
    credit_models[name] = model
metric_table = pd.DataFrame(metric_rows).set_index("model")
print("훈련·테스트:", X_train.shape, X_test.shape)
print(metric_table.round(4))
print("RandomForest 혼동행렬 (행=실제, 열=예측, 순서=0·1):")
print(confusion_matrix(y_test, credit_models["RandomForest balanced"].predict(X_test), labels=[0, 1]))

훈련 24,000명, 테스트 6,000명입니다. `zero_division=0`은 양성을 하나도 예측하지 않아 precision 계산 분모가 0이면 0으로 기록합니다. 기준 모델의 recall과 F1이 0인 이유를 설명합니다. `class_weight="balanced"`는 훈련 시 클래스별 가중치를 조정하며, 부도율을 바꾸거나 예측을 보장하지 않습니다.

**종합 실습:** Logistic Regression의 부도 확률을 0.5 대신 0.3 기준으로 나눠 precision과 recall을 비교합니다. 이는 지표 간 관계를 보는 연습입니다. 운영 기준을 정하려면 별도 검증 자료와 오류 비용 검토가 필요합니다.

<details>
<summary>종합 실습 예시 코드</summary>

```python
probability = credit_models["Logistic Regression"].predict_proba(X_test)[:, 1]
threshold_rows = []
for threshold in [0.3, 0.5]:
    pred = (probability >= threshold).astype(int)
    threshold_rows.append({"threshold": threshold,
                           "precision": precision_score(y_test, pred, zero_division=0),
                           "recall": recall_score(y_test, pred, zero_division=0),
                           "F1": f1_score(y_test, pred, zero_division=0)})
print(pd.DataFrame(threshold_rows).round(4))
```

</details>

임계값 0.3·0.5를 테스트 자료에서 비교하는 코드는 지표의 변화를 보는 교육용 시연입니다. 테스트 점수가 좋아지는 임계값을 그대로 운영 기준으로 고르지 않습니다. 실제 기준은 오류 비용과 별도 검증 자료로 정하고 테스트는 마지막 평가에 사용합니다.

## 확인 문제

1. Target 1은 과거 연체인가요, 다음 달 부도인가요?
2. 전체 정확도 약 78%만으로 부도 예측을 잘한다고 말할 수 있나요?
3. 기준 확률을 낮추면 어떤 오류가 늘어날 수 있나요?

<details>
<summary>정답</summary>

1. 다음 달 부도입니다. 과거 연체 이력은 입력 정보입니다.
2. 아닙니다. 비부도를 항상 고르는 모델도 비슷한 정확도를 얻고 부도는 하나도 찾지 못합니다.
3. 실제 비부도 고객을 부도로 예측하는 거짓 양성이 늘어날 수 있습니다. precision과 recall을 함께 확인합니다.

</details>

## 직접 해보기와 정답

혼동행렬의 네 칸을 이름으로 읽고 모델이 놓친 부도 고객 수를 확인합니다.

In [ ]:
pred = credit_models["RandomForest balanced"].predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
print("실제 비부도 / 비부도 예측:", tn)
print("실제 비부도 / 부도 예측:", fp)
print("실제 부도 / 비부도 예측:", fn)
print("실제 부도 / 부도 예측:", tp)
print("네 칸 합계:", tn + fp + fn + tp)

질문: 부도를 항상 0으로 예측한 기준 모델의 Accuracy가 약 78%인데도 부족한 이유를 한 문장으로 쓰세요.

<details>
<summary>정답</summary>

실제 부도 고객을 하나도 찾지 못해 Recall과 F1이 0이기 때문입니다. 혼동행렬 네 칸의 합은 테스트 6,000명과 같아야 합니다. 실제 부도를 비부도로 예측한 FN이 커지면 부도 고객을 놓치는 사례가 많아집니다. 어떤 오류를 더 줄여야 하는지는 업무 목적과 오류 비용에 따라 달라지며, 이 수업에서는 지표를 읽는 연습을 합니다.

</details>

## 흔한 오류와 해결

| 증상 | 해결 |
| --- | --- |
| Target 열 KeyError | header=1로 읽었는지, columns 출력에 실제 열 이름이 있는지 확인합니다. |
| PAY_1 KeyError | 이 파일에는 PAY_1이 없으며 최근 월은 PAY_0입니다. |
| Titanic 결과가 다시 나옴 | 새 Notebook을 쓰고 credit 입력으로 X·y를 다시 만드는 셀을 실행합니다. |
| Precision 분모가 0 | 양성을 하나도 예측하지 않은 경우입니다. zero_division=0으로 0을 기록하고 Recall도 함께 봅니다. |

## 완료 체크

- [ ] 수업 폴더에서 uv로 Notebook을 열고 Kernel의 `sys.executable`이 수업 `.venv`인지 확인했습니다. [환경 안내](surl:14892)
- [ ] 실습을 저장한 뒤 Kernel을 재시작하고 선행 페이지의 준비 셀부터 위에서 아래로 다시 실행했습니다. 웹 수집은 저장한 자료와 성공·실패 기록을 구분했습니다.

- [ ] 부도 6,636명·비부도 23,364명과 22.12% 부도율을 확인했습니다.
- [ ] ID·Target을 입력에서 제외했습니다.
- [ ] 네 평가 지표와 혼동행렬을 해석했습니다.
- [ ] 확률 기준을 내릴 때 Recall과 Precision의 변화를 확인했습니다.

## 기준 자료

공식 안내문 3쪽의 「금융 데이터 분석 및 신용카드 부도 예측」 2시간를 기준으로 구성했습니다. 기존 단원 자료의 실습 코드를 교시별로 나눈 초안입니다. 실행 결과의 행 수는 제공 파일을 기준으로 하며, 라이브 수집 자료나 라이브러리 버전이 달라지면 값이 달라질 수 있습니다.

<!-- period-navigation:start -->
## 교시별 이동

[교시별 전체 안내](surl:14891) · [공식 단원별 안내](surl:14880) · [이전: 10월 7일 6교시 — Titanic 분류 모델 비교](surl:14900) · [다음: 10월 8일 1~2교시 — 주가 수집과 특성 만들기](surl:14902)
<!-- period-navigation:end -->